# Catastrophic Forgetting and LoRA Merging

Fine-tuning a model on a new task risks degrading its performance on tasks it previously handled well. This is catastrophic forgetting — a fundamental challenge in continual learning. LoRA-based fine-tuning reduces but does not eliminate this risk.

## Why Forgetting Happens

Neural networks store knowledge in their weights as overlapping distributed representations. Fine-tuning on a new task updates weights that were previously tuned for other capabilities. If the gradient signal from the new task conflicts with the representations needed for old tasks, performance degrades.

In full fine-tuning, this is severe: fine-tuning GPT-3 on medical QA noticeably degraded its coding ability. In LoRA, the damage is reduced because:
1. The base weights are frozen — only the low-rank adapters change
2. The adapter's limited capacity constrains how much it can overwrite base knowledge
3. The LoRA update is additive — it shifts the output distribution but does not destroy the base features

However, LoRA fine-tuning can still cause capability regression if the fine-tuning data distribution is narrow or if training continues too long.

In [ ]:
import random
import math
from dataclasses import dataclass, field

@dataclass
class CapabilityScore:
    task: str
    before_ft: float
    after_ft: float
    change: float
    regression: bool

def simulate_forgetting(
    capabilities: dict,
    ft_task: str,
    ft_epochs: int = 3,
    lora_rank: int = 16,
    seed: int = 42
) -> list:
    rng = random.Random(seed)
    results = []
    # Higher rank = more expressive but more forgetting risk
    forgetting_scale = lora_rank / 64  # normalized to rank 64
    for task, base_score in capabilities.items():
        before = base_score
        if task == ft_task:
            # Target task improves
            gain = 0.15 * ft_epochs * (1 - base_score) * rng.uniform(0.8, 1.2)
            after = min(1.0, before + gain)
        else:
            # Other tasks may regress based on task similarity and rank
            # More similar to ft_task = more forgetting
            similarity = rng.uniform(0, 0.3)
            forgetting = similarity * forgetting_scale * ft_epochs * 0.05 * rng.uniform(0.5, 1.5)
            after = max(0, before - forgetting)
        change = after - before
        results.append(CapabilityScore(
            task=task, before_ft=round(before, 3), after_ft=round(after, 3),
            change=round(change, 3), regression=change < -0.02
        ))
    return results

base_capabilities = {
    'math_reasoning': 0.72,
    'code_generation': 0.68,
    'summarization': 0.81,
    'instruction_following': 0.75,
    'creative_writing': 0.70,
    'medical_qa': 0.45,  # fine-tuning target
}

results = simulate_forgetting(base_capabilities, ft_task='medical_qa', lora_rank=16)
print(f'Fine-tuning on medical_qa (LoRA r=16, 3 epochs):')
print(f'{'Task':<25} {'Before':>8} {'After':>8} {'Change':>8} {'Regression'}')
print('-' * 60)
for r in results:
    flag = ' <-- REGRESSION' if r.regression else ''
    print(f'{r.task:<25} {r.before_ft:>8.3f} {r.after_ft:>8.3f} {r.change:>+8.3f}{flag}')

## LoRA Merging Strategies

When you have multiple LoRA adapters (one per task), you can merge them into a single model using different strategies:

**Linear interpolation (LoRA-Mix)**: merge two adapters by averaging their weights with a mixing coefficient: `A_merged = λ*A1 + (1-λ)*A2`. Simple but ignores parameter interactions.

**TIES-Merging**: resolves sign conflicts between adapters before averaging — only merge parameters where both adapters agree on the direction of change.

**DARE** (Drop and Rescale): randomly drop a fraction of delta weights, then rescale the remaining ones. Reduces interference between merged adapters.

**Model arithmetic**: treat merged models as vectors in weight space. Adding adapters adds capabilities; subtracting removes them. This enables capability vectors.

In [ ]:
def lora_linear_merge(adapter_a: dict, adapter_b: dict, alpha: float = 0.5) -> dict:
    merged = {}
    all_keys = set(adapter_a) | set(adapter_b)
    for key in all_keys:
        if key in adapter_a and key in adapter_b:
            merged[key] = alpha * adapter_a[key] + (1 - alpha) * adapter_b[key]
        elif key in adapter_a:
            merged[key] = alpha * adapter_a[key]
        else:
            merged[key] = (1 - alpha) * adapter_b[key]
    return merged

def dare_merge(adapter_a: dict, adapter_b: dict, drop_rate: float = 0.1, seed: int = 42) -> dict:
    rng = random.Random(seed)
    merged = lora_linear_merge(adapter_a, adapter_b)
    # Drop fraction and rescale
    result = {}
    for key, val in merged.items():
        if rng.random() < drop_rate:
            result[key] = 0.0  # dropped
        else:
            result[key] = val / (1 - drop_rate)  # rescaled
    return result

def evaluate_merge(merged: dict, target_keys: list) -> float:
    # Proxy: what fraction of target task params are non-zero and well-scaled
    target_vals = [merged.get(k, 0) for k in target_keys]
    nonzero = sum(1 for v in target_vals if abs(v) > 0.01)
    return nonzero / len(target_keys) if target_keys else 0

# Simulate two task-specific adapters
adapter_medical = {f'layer{i}.A': random.gauss(0, 0.1) for i in range(10)}
adapter_medical.update({f'layer{i}.B': random.gauss(0, 0.05) for i in range(10)})
adapter_code = {f'layer{i}.A': random.gauss(0, 0.08) for i in range(10)}
adapter_code.update({f'layer{i}.B': random.gauss(0, 0.06) for i in range(10)})

linear = lora_linear_merge(adapter_medical, adapter_code, alpha=0.5)
dare = dare_merge(adapter_medical, adapter_code, drop_rate=0.15)

print('Merge strategy comparison:')
print(f'  Medical adapter avg abs weight: {sum(abs(v) for v in adapter_medical.values())/len(adapter_medical):.4f}')
print(f'  Linear merge avg abs weight:    {sum(abs(v) for v in linear.values())/len(linear):.4f}')
print(f'  DARE merge avg abs weight:      {sum(abs(v) for v in dare.values())/len(dare):.4f}')
print(f'  DARE zero params:               {sum(1 for v in dare.values() if v == 0)}/{len(dare)}')

## Mitigations for Catastrophic Forgetting

**Elastic Weight Consolidation (EWC)**: add a regularization term penalizing large changes to weights that were important for previous tasks (estimated via Fisher information). Computationally expensive but principled.

**Replay buffers**: maintain a small set of examples from previous tasks and include them in each training batch. Empirically effective for LoRA fine-tuning.

**Low-rank regularization**: keep fine-tuning rank low (r=4-8) when forgetting is a concern. Lower rank = less expressive but less overwriting.

**Task-specific adapters without merging**: for production systems where tasks are distinct, keep separate LoRA adapters per task and route requests to the appropriate adapter. No forgetting, but storage and routing overhead.